<a href="https://colab.research.google.com/github/aditya-dwivedi13/RL-TicTacToe/blob/main/Training_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.append('/content/drive/MyDrive')

In [3]:
from tictactoe import Game,Board,Player
import numpy as np
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time, random

In [4]:
class CNNModel(nn.Module):
    def __init__(self,load=False,name=None):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=2,
            out_channels=16,
            kernel_size=2)
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 1)
        if load and name is not None:
           self.load(name)

    def forward(self,state,move):
      x = torch.stack((state,move),dim=0)
      x = x.unsqueeze(0)
      x = F.relu(self.conv1(x))
      x = torch.flatten(x,1)
      x = F.relu(self.fc1(x))
      score = torch.tanh(self.fc2(x))
      return score

    def save(self, name="cnn_model.pth"):
       folder = "/content/drive/MyDrive/models"
       os.makedirs(folder, exist_ok=True)
       path = os.path.join(folder, name)
       torch.save(self.state_dict(), path)
       print(f"Model saved to: {path}")

    def load(self, name):
       folder = "/content/drive/MyDrive/models"
       path = os.path.join(folder, name)
       self.load_state_dict(torch.load(path))
       self.eval()
       print(f"Model loaded from: {path}")

In [5]:
class AIAgent:

    def __init__(self, model,epsilon=1, epsilon_min=0.01, epsilon_decay=0.99993):
        self.model = model
        self.episode = [] #stores the episode where each element is in the form (state,encoded move,predicted score)
        # Below parametres are used to implement some exploration
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

    def decay_epsilon(self):
        '''Also decay the epsilon'''
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def reset_episode(self):
        self.episode.clear()

    def encode_move(self, block,value):
      block = block-0.001
      x = int(block//3)
      y = round(block%3) - 1
      move = torch.zeros((3,3), dtype=torch.float32)
      move[x, y] = value
      return move

    def encode_state(self, board):
      return torch.tensor(board.state,dtype=torch.float32)

    def legal_blocks(self, states):
      # states should be a flatten numpy array
      blocks = [i+1 for i in range(9) if states[i] == 0]
      return blocks

    def choose_move(self, board, encode=True, typ=1):
      ''' This method chooses the move with most score and restricts exploration while training to some extent.
      We intend to implement a method in later versions so that it can start exploration'''
      state = self.encode_state(board)
      best_score = -float("inf")
      best_move = None
      best_pred = None
      for block in self.legal_blocks(board.state.flatten()):
          encoded = self.encode_move(block,value=typ)
          score = self.model(state, encoded)

          if score.item() > best_score:
              best_score = score.item()
              best_pred = score
              if encode:
                best_move= encoded
              else:
                best_move = block

      return best_move,best_pred

    def explore_move(self, board, encode=True, typ=1):
      state = self.encode_state(board)
      legal_moves = self.legal_blocks(board.state.flatten())

      # ---------- Exploration ----------
      if random.random() < self.epsilon:
          block = random.choice(legal_moves)
          encoded = self.encode_move(block, value=typ)
          prediction = self.model(state, encoded)
          if encode:
              return encoded, prediction
          else:
              return block, prediction

      # ---------- Exploitation ----------
      return self.choose_move(board, encode=encode, typ=typ)

In [6]:
class SelfPlay:
  def __init__(self, model) -> None:
     self.game = Game()
     self.model = model
     self.agent1 = AIAgent(self.model)
     self.agent2 = AIAgent(self.model)

  def self_play(self,explore = False):
    self.agent1.reset_episode()
    self.agent2.reset_episode()
    turn = 1
    while True:
      if turn==1:
        player = self.game.player1
        if explore:
          move,pred = self.agent1.explore_move(board=self.game.board,
                      typ=1,
                      encode=True)

        else:
          move,pred = self.agent1.choose_move(board=self.game.board,
                      typ=1,
                      encode=True)
        self.agent1.episode.append((self.agent1.encode_state(self.game.board), move, pred))

      else:
        player = self.game.player2
        if explore:
          move,pred = self.agent2.explore_move(board=self.game.board,
                      typ=-1,
                      encode=True)
        else:
          move,pred = self.agent2.choose_move(board=self.game.board,
                      typ=-1,
                      encode=True)
        self.agent2.episode.append((self.agent2.encode_state(self.game.board), move, pred))

      block = torch.argmax(move.abs()).item() + 1
      cmd = self.game.move(player,block)
      if cmd == "play":
        turn = 3-turn
      else:
        continue

      winner = self.game.checkwin()
      if winner == 1:
        self.game.board.reset()
        return 1
      elif winner == 2:
        self.game.board.reset()
        return -1
      elif self.game.checkdraw():
        self.game.board.reset()
        return 0

In [7]:
class Trainer:
    ''' Trains the model using selfplay and playing one episode followed by one backpropagation at a time for n number of times'''
    def __init__(self, model, gamma=0.9,learning_rate=1e-3,explore=False):
        self.model = model
        self.gamma = gamma
        self.loss_function = nn.MSELoss()
        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=learning_rate)
        self.self_play = SelfPlay(self.model)
        self.explore = explore

    def compute_targets(self, r1, r2):

        episode_agent1 = self.self_play.agent1.episode
        episode_agent2 = self.self_play.agent2.episode

        targets_agent1 = []
        targets_agent2 = []

        for index in range(len(episode_agent1)):
         if index == len(episode_agent1) - 1:
            target = torch.full_like(episode_agent1[index][2].detach(),
    float(r1) )
         else:
            target = (
                    self.gamma *
                    episode_agent1[index + 1][2].detach() )

         targets_agent1.append(target)


        for index in range(len(episode_agent2)):
            if index == len(episode_agent2) - 1:
                target = torch.full_like(episode_agent2[index][2].detach(),float(r2) )
            else:
                target = (
                    self.gamma *
                    episode_agent2[index + 1][2].detach())

            targets_agent2.append(target)


        targets_agent1 = torch.stack(targets_agent1)
        targets_agent2 = torch.stack(targets_agent2)

        return targets_agent1, targets_agent2


    def train(self, episodes=1000, log=100, save=False,name='model_v1.pth'):
        print('Training Started')
        win1,win2,draw,draw_percentage = 0,0,0,90
        for episode in range(episodes):
            winner = self.self_play.self_play(explore=self.explore)
            if winner == 1:
                win1 += 1
            elif winner == -1:
                win2 += 1
            else:
                draw += 1
            r1 = winner
            r2 = -winner
            targets_agent1, targets_agent2 = self.compute_targets(r1,r2)

            predictions_agent1 = torch.stack([experience[2] for experience in self.self_play.agent1.episode])
            predictions_agent2 = torch.stack([experience[2] for experience in self.self_play.agent2.episode])

            loss_agent1 = self.loss_function(
                predictions_agent1,
                targets_agent1 )
            loss_agent2 = self.loss_function(
                predictions_agent2,
                targets_agent2 )

            self.optimizer.zero_grad()
            loss_agent1.backward()
            loss_agent2.backward()
            self.optimizer.step()

            # Decay the epsilon
            self.self_play.agent1.decay_epsilon()
            self.self_play.agent2.decay_epsilon()

            if (episode + 1) % log == 0:
               if save:
                  self.model.save(name=name)
               print()
               print(
                    f"Episode {episode + 1}/{episodes} | "
                    f"Loss: {(loss_agent1 + loss_agent2).item():.4f}"
                )
               print(f"Agent1: {win1*100/log}% | Win2: {win2*100/log}% | Draw: {draw*100/log}%")

               if draw*100/log > draw_percentage:
                  print("Got high draw percentage",draw*100/log)
                  self.model.save(name="optimum.pth")
                  draw_percentage=draw*100/log
               win1,win2,draw = 0,0,0
        if save:
          print('Saving the final model')
          self.model.save(name=name)

In [8]:
model = CNNModel(load=False,name='model_exploring_for_100k.pth')

In [9]:
%%time
trainer = Trainer(model,explore=True)
trainer.train(episodes=150000, log=2000, save=True)

Training Started
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 2000/150000 | Loss: 0.2613
Agent1: 59.65% | Win2: 28.55% | Draw: 11.8%
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 4000/150000 | Loss: 1.0454
Agent1: 61.1% | Win2: 28.9% | Draw: 10.0%
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 6000/150000 | Loss: 0.2255
Agent1: 60.95% | Win2: 30.2% | Draw: 8.85%
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 8000/150000 | Loss: 0.1419
Agent1: 62.05% | Win2: 31.65% | Draw: 6.3%
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 10000/150000 | Loss: 0.8178
Agent1: 64.8% | Win2: 31.0% | Draw: 4.2%
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 12000/150000 | Loss: 0.1104
Agent1: 68.1% | Win2: 27.35% | Draw: 4.55%
Model saved to: /content/drive/MyDrive/models/model_v1.pth

Episode 14000/150000 | Loss: 0.9208
Agent1: 67.6% | Win2: 26.75% | Draw: 5.65%
Model saved to: 

In [10]:
class AIGame:
  def __init__(self,model) -> None:
     self.game = Game()
     self.model = model
     self.agent = AIAgent(self.model)

  def human_move(self):
    block = int(input('Your move enter a block number: '))
    return block

  def ai_move(self,value):
    position = self.agent.choose_move(board = self.game.board, encode=False,typ=value)[0]
    return position

  def play_(self,human=1):
    self.game.board.display()
    turn = 1
    value = -1 if human == 1 else 1
    while True:
      if turn==2:
        player = self.game.player2
      else:
        player = self.game.player1

      if human == turn:
        block = self.human_move()
      else:
        block = self.ai_move(value)
        print('Thinking..')
        time.sleep(2)
        print('AI Move',block)

      cmd = self.game.move(player,block)
      if cmd == "play":
        self.game.board.display()
        turn = 3-turn
      else:
        continue

      r = self.game.checkwin()
      if r==0:
        if self.game.checkdraw():
          print("Draw")
          break
      else:
        print(f"Player{r} Won!!")
        break

In [14]:
agent = CNNModel(load=True, name="optimum.pth")

Model loaded from: /content/drive/MyDrive/models/optimum.pth


In [15]:
game = AIGame(model)
game.play_(human=2)

_ _ _
_ _ _
_ _ _
Thinking..
AI Move 5
_ _ _
_ O _
_ _ _
Your move enter a block number: 1
X _ _
_ O _
_ _ _
Thinking..
AI Move 4
X _ _
O O _
_ _ _
Your move enter a block number: 5
Invalid move
Your move enter a block number: 6
X _ _
O O X
_ _ _
Thinking..
AI Move 2
X O _
O O X
_ _ _
Your move enter a block number: 8
X O _
O O X
_ X _
Thinking..
AI Move 7
X O _
O O X
O X _
Your move enter a block number: 3
X O X
O O X
O X _
Thinking..
AI Move 9
X O X
O O X
O X O
Draw


In [16]:
game = AIGame(model)
game.play_(human=2)

_ _ _
_ _ _
_ _ _
Thinking..
AI Move 5
_ _ _
_ O _
_ _ _
Your move enter a block number: 8
_ _ _
_ O _
_ X _
Thinking..
AI Move 1
O _ _
_ O _
_ X _
Your move enter a block number: 9
O _ _
_ O _
_ X X
Thinking..
AI Move 2
O O _
_ O _
_ X X
Your move enter a block number: 7
O O _
_ O _
X X X
Player2 Won!!


In [17]:
game = AIGame(model)
game.play_(human=2)

_ _ _
_ _ _
_ _ _
Thinking..
AI Move 5
_ _ _
_ O _
_ _ _
Your move enter a block number: 2
_ X _
_ O _
_ _ _
Thinking..
AI Move 1
O X _
_ O _
_ _ _
Your move enter a block number: 3
O X X
_ O _
_ _ _
Thinking..
AI Move 9
O X X
_ O _
_ _ O
Player1 Won!!


In [18]:
game = AIGame(model)
game.play_(human=1)

_ _ _
_ _ _
_ _ _
Your move enter a block number: 5
_ _ _
_ O _
_ _ _
Thinking..
AI Move 2
_ X _
_ O _
_ _ _
Your move enter a block number: 1
O X _
_ O _
_ _ _
Thinking..
AI Move 9
O X _
_ O _
_ _ X
Your move enter a block number: 4
O X _
O O _
_ _ X
Thinking..
AI Move 6
O X _
O O X
_ _ X
Your move enter a block number: 8
O X _
O O X
_ O X
Thinking..
AI Move 3
O X X
O O X
_ O X
Player2 Won!!


In [20]:
game = AIGame(model)
game.play_(human=2)

_ _ _
_ _ _
_ _ _
Thinking..
AI Move 5
_ _ _
_ O _
_ _ _
Your move enter a block number: 6
_ _ _
_ O X
_ _ _
Thinking..
AI Move 1
O _ _
_ O X
_ _ _
Your move enter a block number: 9
O _ _
_ O X
_ _ X
Thinking..
AI Move 2
O O _
_ O X
_ _ X
Your move enter a block number: 3
O O X
_ O X
_ _ X
Player2 Won!!


Without Exploration the model when initialised makes one agent win more over the other one and upon training it pushes only one to win more to such an extent that one agent gets 100% win rate that means it somehow gets trapped in a policy that forces on agent to win (that too mostly agent 1).
#
With Epsilon greedy Exploration the models gets freedom to explore various other policies even if one policy.
#
Here some amusing pattern was seen from the training logs inference that first there are a lot of fluctuations but still a general pattern was observed that goes like firstly the model intialises on one policy that let's say results  agent1 to win more and agent2 to lose more so then it will keep on training this policy until it does it exceptionally well like agent1 win percentages goes up 98 or even 99 (but never hundred) and then it will suddenly change its policy and thus drop the value of agent1 win percentage to very low values (change is very sudden eg. Agent1 win% will go from something like 97% to straight up 95%) and then keep on improving this strategy and then suddenly drop it too and repeat this whole process for multiple times until the training ends
#
We can use this pattern and when the model falls in a policy that increases number of draws then we can save a version of the model (Optimum) right there.

# Optimum V1
### Draw Rate : 97.15%
#
### Learnt To :-
#### . To block opponent from direct wins but not forks
#### . To Hold Centre as soon as possible
#### . To try and finish the game if already has 2 cells in one line
#### . Also tries to make it's lines

### Improvements :-
#### . Can't foresee forks
#### . Doesn't know how to use the Centre to fork and if the opponent creates a situation where if you block them then you create a fork and can easily and if you don't block them they can win easily in such cases the agent still chooses not to block ( to avoid the fork maybe ig)